In [1]:
import torch
import torch.nn as nn 
from torch.nn import functional as F
torch.manual_seed(42)

In [3]:
# Downloading the tiny shakespeare dataset
!curl -sL -o input.txt https://raw.githubusercontent.com/AviSoori1x/makeMoE/main/input.txt

In [4]:
#Expert module
class Expert(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)
    

In [7]:
num_experts = 3
top_k = 2
n_embd = 8

#Example multi latent attention output for a simple illustrative example, consider n_embd=8
mh_output = torch.randn(1, 4, n_embd)
topkgate_linear = nn.Linear(n_embd, num_experts) #Linear(8,2)
logits = topkgate_linear(mh_output)
print(logits)

tensor([[[-0.5033, -0.1011, -0.7712],
         [-0.4743, -0.5012, -0.3763],
         [ 0.1822, -0.2049, -0.1416],
         [-0.3654, -0.5363,  0.0282]]], grad_fn=<ViewBackward0>)


## Implementent topk load balanacing

In [8]:
top_k_logits, top_k_indices = logits.topk(top_k, dim=-1)
top_k_logits , top_k_indices

(tensor([[[-0.1011, -0.5033],
          [-0.3763, -0.4743],
          [ 0.1822, -0.1416],
          [ 0.0282, -0.3654]]], grad_fn=<TopkBackward0>),
 tensor([[[1, 0],
          [2, 0],
          [0, 2],
          [2, 0]]]))

## Use -inf and apply softmax

In [9]:
zeros = torch.full_like(logits, float('-inf'))
sparse_logits = zeros.scatter(-1, top_k_indices, top_k_logits)
sparse_logits

tensor([[[-0.5033, -0.1011,    -inf],
         [-0.4743,    -inf, -0.3763],
         [ 0.1822,    -inf, -0.1416],
         [-0.3654,    -inf,  0.0282]]], grad_fn=<ScatterBackward0>)

In [10]:
gating_output = F.softmax(sparse_logits, dim=-1)
gating_output

tensor([[[0.4008, 0.5992, 0.0000],
         [0.4755, 0.0000, 0.5245],
         [0.5802, 0.0000, 0.4198],
         [0.4029, 0.0000, 0.5971]]], grad_fn=<SoftmaxBackward0>)

## Creating a class for TopKRouting

In [ ]:
class TopKRouter(nn.Module):
    def __init__(self, n_embd, num_experts, top_k):
        super(TopKRouter, self).__init__()
        self.top_K = top_k
        self.linear = nn.Linear(n_embd, num_experts)

    def forward(self, mh_output):
        logits 